# Amazon UK Product Analysis

**Objective:** Explore the product listing dynamics on Amazon UK to extract actionable business insights.

**Dataset:** Place the CSV file named `amazon_uk_products.csv` in the notebook working directory (`/workspace` or the folder where you run this notebook). If your filename or path differs, update the `DATA_PATH` variable in the first code cell.

---

### How to use this notebook

1. Download the dataset and save it as `amazon_uk_products.csv` in the same folder as this notebook.
2. Run the cells sequentially. The notebook includes code for data loading, cleaning, exploratory analysis, visualizations, and a final business-centric summary.
3. If you do not have the CSV locally but have a URL to it, update `DATA_URL` in the first code cell and uncomment the download snippet.



In [ ]:
# Setup: imports and data path - update DATA_PATH if necessary
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 200)
plt.rcParams['figure.figsize'] = (10,6)

# Update these if your file is elsewhere or has a different name
DATA_PATH = 'amazon_uk_products.csv'  # <-- put the dataset here
# Optional: if you have a direct URL to the CSV, set DATA_URL and uncomment the download block below
DATA_URL = ''

# Optional download (uncomment if you set DATA_URL)
# import requests
# r = requests.get(DATA_URL)
# open(DATA_PATH, 'wb').write(r.content)

print('Notebook ready. Make sure the dataset file exists at:', os.path.abspath(DATA_PATH))

In [ ]:
# Load dataset
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Dataset not found at {DATA_PATH}. Place the CSV there or update DATA_PATH.")

df = pd.read_csv(DATA_PATH)
print('Loaded dataset with shape:', df.shape)
df.head()

In [ ]:
# Basic cleaning / overview
# Inspect columns and dtypes
display(df.dtypes)
print('\nSample of columns:', list(df.columns)[:40])

# Common cleaning steps - adapt depending on actual dataset column names
# Try to locate price and rating columns by likely names
possible_price_cols = [c for c in df.columns if 'price' in c.lower() or 'advised' in c.lower() or 'amount' in c.lower()]
possible_rating_cols = [c for c in df.columns if 'rating' in c.lower() or 'stars' in c.lower()]

print('\nPossible price columns:', possible_price_cols)
print('Possible rating columns:', possible_rating_cols)

# If price column is not numeric, attempt to clean it (strip currency symbols)
def clean_price_series(s):
    s = s.astype(str).str.replace('[^0-9\.\-]', '', regex=True)
    s = s.replace('', np.nan)
    return pd.to_numeric(s, errors='coerce')

# Example: try to pick a price and rating column automatically if present
price_col = possible_price_cols[0] if possible_price_cols else None
rating_col = possible_rating_cols[0] if possible_rating_cols else None

print('\nAuto-selected price column:', price_col)
print('Auto-selected rating column:', rating_col)

if price_col:
    df['price_clean'] = clean_price_series(df[price_col])
if rating_col:
    df['rating_clean'] = pd.to_numeric(df[rating_col], errors='coerce')

print('\nAfter cleaning: price_clean stats:')
if 'price_clean' in df.columns:
    display(df['price_clean'].describe())
if 'rating_clean' in df.columns:
    display(df['rating_clean'].describe())

## Part 1: Understanding Product Categories

**Business question:** What are the most popular product categories on Amazon UK, and how do they compare in terms of listing frequency?

**Steps:** frequency table, top 5 categories, bar chart for distribution, pie chart for top categories.


In [ ]:
# Adjust category column name if your dataset uses different naming.
possible_category_cols = [c for c in df.columns if 'category' in c.lower() or 'browse' in c.lower() or 'department' in c.lower()]
possible_category_cols[:5]

In [ ]:
# Choose a category column automatically if found
category_col = possible_category_cols[0] if possible_category_cols else None
if category_col is None:
    raise ValueError('No category-like column found automatically. Please inspect df.columns and set category_col manually.')
print('Using category column:', category_col)

# Frequency table
category_counts = df[category_col].fillna('Unknown').astype(str).value_counts()
category_counts.name = 'count'
display(category_counts.head(20))

# Top 5 categories
top5 = category_counts.head(5)
print('\nTop 5 categories:')
display(top5)

# Bar chart (top N)
top_n = 20
topn = category_counts.head(top_n)
plt.figure(figsize=(12,6))
topn.plot.bar()
plt.title(f'Top {top_n} product categories by listing frequency')
plt.ylabel('Number of listings')
plt.xlabel('Category')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Pie chart for top categories (top 5)
plt.figure(figsize=(8,8))
top5.plot.pie(autopct='%1.1f%%', startangle=140)
plt.ylabel('')
plt.title('Proportion of top 5 categories')
plt.show()

## Part 2: Delving into Product Pricing

**Business question:** How are products priced on Amazon UK, and are there specific price points or ranges that are more common?



In [ ]:
# Ensure price_clean exists
if 'price_clean' not in df.columns:
    raise ValueError('No price_clean column found. Check earlier cleaning steps and set price_col correctly.')

price = df['price_clean'].dropna()
print('Price series length:', len(price))

# Measures of centrality
mean_price = price.mean()
median_price = price.median()
mode_price = price.mode()
mode_price_val = mode_price.iloc[0] if not mode_price.empty else np.nan
print('Mean price:', mean_price)
print('Median price:', median_price)
print('Mode price (one of possibly several):', mode_price_val, 'Full mode series:', list(mode_price.values))

# Measures of dispersion
variance = price.var()
std_dev = price.std()
price_range = price.max() - price.min()
iqr = price.quantile(0.75) - price.quantile(0.25)

print('\nVariance:', variance)
print('Standard deviation:', std_dev)
print('Range:', price_range)
print('IQR:', iqr)

# Histogram (log scale option if distribution is skewed)
plt.figure(figsize=(10,6))
plt.hist(price, bins=60)
plt.title('Histogram of product prices (raw scale)')
plt.xlabel('Price')
plt.ylabel('Number of products')
plt.show()

# If distribution is heavily skewed, plot log price histogram
if (price.skew() > 2) or (price.max() / (price.median()+1e-9) > 10):
    plt.figure(figsize=(10,6))
    plt.hist(np.log1p(price), bins=60)
    plt.title('Histogram of log(1 + price) (to show skewed distribution)')
    plt.xlabel('log(1 + price)')
    plt.ylabel('Number of products')
    plt.show()

# Boxplot to find outliers
plt.figure(figsize=(10,4))
plt.boxplot(price.dropna(), vert=False)
plt.title('Boxplot of product prices (raw scale)')
plt.xlabel('Price')
plt.show()

## Part 3: Unpacking Product Ratings

**Business question:** How do customers rate products on Amazon UK, and are there any patterns or tendencies in the ratings?



In [ ]:
# Ensure rating_clean exists
if 'rating_clean' not in df.columns:
    print('No rating_clean column found automatically. Attempting to detect another likely column...')
    possible_rating_cols = [c for c in df.columns if 'rating' in c.lower() or 'stars' in c.lower()]
    print('Possible rating columns:', possible_rating_cols)
    if possible_rating_cols:
        df['rating_clean'] = pd.to_numeric(df[possible_rating_cols[0]], errors='coerce')
    else:
        raise ValueError('No rating column found. Please inspect df.columns and set rating column manually.')

ratings = df['rating_clean'].dropna()
print('Number of rated products:', len(ratings))

# Measures of centrality
mean_rating = ratings.mean()
median_rating = ratings.median()
mode_rating = ratings.mode()
mode_rating_val = mode_rating.iloc[0] if not mode_rating.empty else np.nan
print('Mean rating:', mean_rating)
print('Median rating:', median_rating)
print('Mode rating (one of possibly several):', mode_rating_val, 'Full mode series:', list(mode_rating.values))

# Measures of dispersion
variance_rating = ratings.var()
std_rating = ratings.std()
iqr_rating = ratings.quantile(0.75) - ratings.quantile(0.25)
print('\nVariance (rating):', variance_rating)
print('Std dev (rating):', std_rating)
print('IQR (rating):', iqr_rating)

# Shape: skewness and kurtosis
skewness = ratings.skew()
kurtosis = ratings.kurtosis()  # pandas' fisher default (excess kurtosis)
print('\nSkewness:', skewness)
print('Kurtosis (excess):', kurtosis)

# Histogram of ratings
plt.figure(figsize=(8,5))
plt.hist(ratings, bins=20)
plt.title('Histogram of product ratings')
plt.xlabel('Rating')
plt.ylabel('Number of products')
plt.show()

# Boxplot of ratings
plt.figure(figsize=(8,3))
plt.boxplot(ratings.dropna(), vert=False)
plt.title('Boxplot of product ratings')
plt.xlabel('Rating')
plt.show()

## Part 4: Extra analyses & Business Insights

Suggested next steps and quick code snippets to help generate actionable recommendations.


In [ ]:
# 1) Top brands by average rating (if brand column exists)
possible_brand_cols = [c for c in df.columns if 'brand' in c.lower() or 'manufacturer' in c.lower()]
brand_col = possible_brand_cols[0] if possible_brand_cols else None
if brand_col and 'rating_clean' in df.columns:
    brand_stats = df.groupby(brand_col).agg(
        count_listings = ('rating_clean', 'count'),
        avg_rating = ('rating_clean', 'mean'),
        median_rating = ('rating_clean', 'median')
    ).sort_values('count_listings', ascending=False)
    display(brand_stats.head(10))
else:
    print('No brand column detected or no ratings available to compute brand stats.')

# 2) Price buckets and counts
if 'price_clean' in df.columns:
    bins = [0, 5, 20, 50, 100, 250, 500, 1000, np.inf]
    labels = ['0-5','5-20','20-50','50-100','100-250','250-500','500-1000','1000+']
    df['price_bucket'] = pd.cut(df['price_clean'], bins=bins, labels=labels)
    bucket_counts = df['price_bucket'].value_counts().sort_index()
    display(bucket_counts)
    bucket_counts.plot.bar()
    plt.title('Product counts by price bucket')
    plt.xlabel('Price bucket')
    plt.ylabel('Number of products')
    plt.show()
else:
    print('No price_clean column to create price buckets.')

# 3) Correlation between price and rating (if both exist)
if 'price_clean' in df.columns and 'rating_clean' in df.columns:
    corr = df[['price_clean','rating_clean']].dropna().corr().iloc[0,1]
    print('Pearson correlation between price and rating:', corr)
    plt.figure(figsize=(8,6))
    plt.scatter(df['price_clean'], df['rating_clean'], alpha=0.3)
    plt.xscale('symlog')  # symlog handles zero/very small values for visual clarity
    plt.xlabel('Price')
    plt.ylabel('Rating')
    plt.title('Scatter: price vs rating (symlog x-axis)')
    plt.show()
else:
    print('Need both price_clean and rating_clean to compute correlation.')

### End of notebook

Run the notebook cells and interpret the output. Use the visualizations and metrics to write a business-centric summary (e.g., which categories dominate, whether prices are skewed, whether ratings cluster at high values, and recommended actions such as focusing on top categories, re-pricing strategies, or inventory concentration).
